# 🎓 Parcial – Predicción de deserción estudiantil (ANN / MLP)
**Inteligencia Artificial – Deep Learning** · Dataset: *Predict Students' Dropout and Academic Success*

**Cómo usar este cuaderno en Google Colab**
1. Menú **Entorno de ejecución → Ejecutar todo** (o ejecuta celda por celda en orden).
2. En la celda 3 el dataset se descarga automáticamente de Kaggle; si falla, te pedirá subir el CSV manualmente.
3. Al final se levanta la app Streamlit con un enlace público temporal y se genera un `.zip` para tu repositorio de GitHub.

No necesitas GPU: es un problema tabular pequeño y entrena en segundos en CPU.

**Contenido:** 1) Instalación · 2) Preprocesamiento · 3) EDA · 4) Partición y desbalance · 5) Justificación de arquitectura · 6) Modelo y entrenamiento · 7) Evaluación (umbral, baselines, CV) · 8) Exportación · 9) App web

## 1. Instalación y configuración

In [ ]:
!pip -q install streamlit kagglehub

In [ ]:
import os, glob, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, precision_recall_fscore_support,
                             recall_score, precision_score, f1_score)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)   # reproducibilidad
sns.set_theme(style="whitegrid")
import sklearn
print("TensorFlow:", tf.__version__, "| scikit-learn:", sklearn.__version__)   # anótalas: se fijan en requirements.txt

## 2. Módulo de preprocesamiento (`preprocessing.py`)
Escribimos el preprocesamiento en **un solo archivo** que usarán el entrenamiento *y* la app web. Así garantizamos que
la app transforma los datos **exactamente igual** que en el entrenamiento (evita el *training-serving skew*).

Decisiones de diseño:
- **Categóricas codificadas como número** (estado civil, modo de admisión, programa, formación previa, nacionalidad,
  estudios y ocupación de los padres): son *códigos*, no cantidades; que el programa 15 sea "mayor" que el 2 no significa
  nada. Se aplica **One-Hot Encoding**.
- **Binarias 0/1** (género, becario, deudor, etc.): ya están en formato adecuado, se dejan igual.
- **Continuas/conteos** (notas, edad, unidades curriculares, macroeconomía): se tratan los outliers con *winsorizing*
  (percentiles 1–99 aprendidos solo en train) y luego se aplica **StandardScaler**.
- Todo se ajusta (`fit`) **solo con entrenamiento** para evitar *data leakage*.

In [ ]:
%%writefile preprocessing.py
"""
preprocessing.py
----------------
Definición ÚNICA del preprocesamiento. Lo importan tanto train_model.py como app.py,
así garantizamos que la app transforma los datos EXACTAMENTE igual que en el entrenamiento.
"""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "Target"

# Variables que vienen como números pero son CATEGORÍAS nominales (códigos, no cantidades)
CATEGORICAL = [
    "Marital status", "Application mode", "Course", "Previous qualification",
    "Nacionality", "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation",
]

# Variables binarias 0/1: ya son "one-hot" de por sí, se dejan tal cual
BINARY = [
    "Daytime/evening attendance", "Displaced", "Educational special needs", "Debtor",
    "Tuition fees up to date", "Gender", "Scholarship holder", "International",
]

# Variables numéricas continuas / conteos: se tratan outliers y se escalan
CONTINUOUS = [
    "Application order",
    "Age at enrollment",
    "Curricular units 1st sem (credited)", "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)", "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)", "Curricular units 1st sem (without evaluations)",
    "Curricular units 2nd sem (credited)", "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (evaluations)", "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)", "Curricular units 2nd sem (without evaluations)",
    "Unemployment rate", "Inflation rate", "GDP",
]

FEATURES = CATEGORICAL + BINARY + CONTINUOUS


def load_data(path: str) -> pd.DataFrame:
    """Carga el CSV (detecta el separador ',' o ';') y limpia los nombres de columna."""
    df = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")  # utf-8-sig elimina el BOM
    # limpiar nombres: quita BOM (\ufeff), espacios y tabuladores
    df.columns = [c.replace("\ufeff", "").strip() for c in df.columns]
    faltantes = [c for c in FEATURES + [TARGET] if c not in df.columns]
    if faltantes:
        raise ValueError(f"Columnas no encontradas en el CSV: {faltantes}\n"
                         f"Columnas disponibles: {list(df.columns)}")
    return df


def to_numeric_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Fuerza tipo float en todas las features (mismo tipo en entrenamiento y en la app)."""
    out = df[FEATURES].copy()
    return out.apply(pd.to_numeric, errors="coerce").astype(float)


class Winsorizer(BaseEstimator, TransformerMixin):
    """
    Recorta (clip) los valores extremos a los percentiles [low, high] APRENDIDOS EN TRAIN.
    Así los outliers no distorsionan el StandardScaler y no se pierden filas.
    """
    def __init__(self, low=1, high=99):
        self.low = low
        self.high = high

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lower_ = np.nanpercentile(X, self.low, axis=0)
        self.upper_ = np.nanpercentile(X, self.high, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)


def build_preprocessor() -> ColumnTransformer:
    """ColumnTransformer: imputación + One-Hot (categóricas) | passthrough (binarias) | winsor + scaler."""
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        # min_frequency agrupa categorías raras (evita cientos de columnas casi vacías);
        # handle_unknown evita errores si la app recibe un código nunca visto.
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=15, sparse_output=False)),
    ])
    bin_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent"))])
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("winsor", Winsorizer(1, 99)),
        ("scaler", StandardScaler()),
    ])
    return ColumnTransformer([
        ("cat", cat_pipe, CATEGORICAL),
        ("bin", bin_pipe, BINARY),
        ("num", num_pipe, CONTINUOUS),
    ])

In [ ]:
from preprocessing import (load_data, to_numeric_frame, build_preprocessor,
                           CATEGORICAL, BINARY, CONTINUOUS, FEATURES, TARGET)
print(len(CATEGORICAL), "categóricas |", len(BINARY), "binarias |", len(CONTINUOUS), "continuas")

## 3. Carga de datos y Análisis Exploratorio (EDA)

In [ ]:
# --- Carga automática desde Kaggle; si falla, subir el CSV manualmente ---
DATA_PATH = None
try:
    import kagglehub
    d = kagglehub.dataset_download("thedevastator/higher-education-predictors-of-student-retention")
    candidatos = []
    for f in glob.glob(os.path.join(d, "**", "*.csv"), recursive=True):
        cols = [c.strip() for c in pd.read_csv(f, sep=None, engine="python", nrows=5).columns]
        if "Target" in cols:                      # nos quedamos con el CSV que tiene la variable objetivo
            candidatos.append((os.path.getsize(f), f))
    DATA_PATH = max(candidatos)[1]                # el más grande (dataset completo)
    print("Dataset descargado:", DATA_PATH)
except Exception as e:
    print("No se pudo descargar automáticamente:", e)

if DATA_PATH is None:
    from google.colab import files
    print("Sube el archivo CSV del dataset:")
    up = files.upload()
    DATA_PATH = list(up.keys())[0]

In [ ]:
df = load_data(DATA_PATH)
df[TARGET] = df[TARGET].str.strip()
print("Dimensiones:", df.shape)
display(df.head())
df.info()

In [ ]:
# --- Calidad de datos ---
print("Valores faltantes totales:", int(df.isnull().sum().sum()))
print("Filas duplicadas:", int(df.duplicated().sum()))

# --- Distribución de la variable objetivo (desbalance) ---
conteo = df[TARGET].value_counts()
print(pd.DataFrame({"n": conteo, "%": (100 * conteo / len(df)).round(1)}))
plt.figure(figsize=(5, 3.5))
sns.countplot(data=df, x=TARGET, order=conteo.index, palette="viridis")
plt.title("Distribución de la clase objetivo"); plt.show()

In [ ]:
# --- Detección de outliers con el criterio IQR (1.5 * IQR) ---
filas = []
for c in CONTINUOUS:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    n = int(((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum())
    filas.append((c, n, round(100 * n / len(df), 1)))
display(pd.DataFrame(filas, columns=["variable", "n_outliers", "% filas"]).sort_values("n_outliers", ascending=False).head(10))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, c in zip(axes, ["Age at enrollment", "Curricular units 1st sem (approved)",
                        "Curricular units 1st sem (grade)", "Curricular units 2nd sem (approved)"]):
    sns.boxplot(y=df[c], ax=ax, color="steelblue"); ax.set_title(c, fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# --- Rendimiento académico vs. clase (la señal más fuerte del dataset) ---
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.boxplot(data=df, x=TARGET, y="Curricular units 2nd sem (approved)", ax=axes[0], palette="viridis")
sns.boxplot(data=df, x=TARGET, y="Curricular units 2nd sem (grade)", ax=axes[1], palette="viridis")
plt.tight_layout(); plt.show()

# --- Correlación entre variables continuas ---
plt.figure(figsize=(10, 7))
sns.heatmap(df[CONTINUOUS].corr(), cmap="coolwarm", center=0, xticklabels=False)
plt.title("Correlación entre variables continuas"); plt.show()

**Lectura del EDA (para el informe):** (1) el dataset no tiene valores faltantes ni (normalmente) duplicados; (2) la clase
*Graduate* es la mayoritaria (~50 %), *Dropout* ~32 % y *Enrolled* ~18 %: hay desbalance moderado; (3) hay outliers
en edad y en conteos de unidades curriculares, que son valores **reales** (estudiantes mayores, etc.), por eso
no se eliminan filas: se *recortan* (winsorizing) para que no dominen el escalado; (4) las variables del 1.º y 2.º semestre
están muy correlacionadas entre sí (multicolinealidad), algo que una red con regularización tolera bien.
(5) **Limitación temporal:** las variables del 2.º semestre solo existen al *terminar* ese semestre; el modelo es por tanto un
sistema de alerta «a mitad de carrera del primer año». Una versión *temprana* (solo datos de matrícula + 1.º semestre) sería
menos precisa pero permitiría intervenir antes; se menciona como trabajo futuro.

## 4. Partición estratificada y tratamiento del desbalance
Partición **70 % / 15 % / 15 %** con `stratify` para que cada subconjunto conserve la proporción de clases.

**Desbalance → `class_weight='balanced'`.** Se escogió esta técnica en vez de SMOTE porque, tras el One-Hot, la mayoría de las
columnas son categóricas/binarias y SMOTE interpola entre vecinos generando valores *no válidos* (p. ej. 0.37 de "género").
Con `class_weight` no se inventan datos: la función de pérdida penaliza más los errores en la clase minoritaria
(peso ∝ n_total / (n_clases · n_clase)).

In [ ]:
X = to_numeric_frame(df)
le = LabelEncoder()
y = le.fit_transform(df[TARGET])                    # Dropout=0, Enrolled=1, Graduate=2
print("Clases:", dict(zip(le.classes_, range(len(le.classes_)))))

# 70 / 15 / 15 estratificado
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

# Preprocesador: fit SOLO con train
pre = build_preprocessor()
Xtr = pre.fit_transform(X_train).astype("float32")
Xva = pre.transform(X_val).astype("float32")
Xte = pre.transform(X_test).astype("float32")

for nombre, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{nombre:5s} n={len(yy):4d}  proporciones:", np.round(np.bincount(yy) / len(yy), 3))
print("Entrada de la red (nº de features tras One-Hot + escalado):", Xtr.shape[1])

# Pesos de clase
pesos = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight = {int(c): float(w) for c, w in zip(np.unique(y_train), pesos)}
print("class_weight:", {le.classes_[k]: round(v, 2) for k, v in class_weight.items()})

## 5. Justificación técnica de la arquitectura

### 5.1 ¿Por qué ANN (MLP) y no CNN?
- **Naturaleza de los datos:** cada estudiante es un **vector de ~35 atributos heterogéneos** (códigos, binarias, notas, tasas
  macroeconómicas) en una tabla. No hay estructura espacial ni secuencial.
- **Las CNN explotan una propiedad que aquí no existe:** el *sesgo inductivo* de una convolución asume **localidad** (píxeles
  vecinos se relacionan) e **invarianza a la traslación** (un patrón vale igual en cualquier posición). En una tabla, el orden de las
  columnas es arbitrario: reordenarlas no cambia el problema, y "edad" no está "cerca" de "GDP". Un filtro deslizante
  mezclaría variables sin relación semántica.
- **El MLP** conecta todas las variables con todas (capas densas), aprende **interacciones no lineales** entre ellas
  (p. ej. *deudor* + *pocas materias aprobadas* + *sin beca*) y termina en una capa **Softmax** de 3 neuronas que entrega probabilidades
  por clase.
- **Tamaño:** con 4.424 muestras, un MLP pequeño (≈ decenas de miles de parámetros) es suficiente y evita sobreajuste; una CNN
  profunda sería innecesariamente compleja.

### 5.2 ¿Por qué entrenar desde cero y no usar Transfer Learning?
- Transfer Learning reutiliza **representaciones aprendidas en otra tarea/dominio** (VGG16, ResNet50, MobileNetV2 y EfficientNet
  se entrenaron en **ImageNet**, con imágenes RGB de 224×224). Sus filtros detectan bordes, texturas y formas; **ninguno de
  esos conocimientos se transfiere** a notas, estados civiles o códigos de programa.
- **Incompatibilidad de entrada:** esos modelos exigen tensores `(alto, ancho, 3)`; nuestros datos son un vector `(n_features,)`.
  Convertir la fila en una "imagen" artificial no aporta información y destruye semántica.
- **No existe un modelo preentrenado del tipo VGG/ResNet aplicable** a datos tabulares con este esquema de columnas, que es propio del dataset
  (distinto en cada institución). Existen enfoques recientes de modelos fundacionales tabulares (p. ej. TabPFN), pero no son
  *Transfer Learning* en el sentido clásico de reutilizar filtros y se salen del alcance del parcial. Además, el dataset es pequeño pero la red también, así que **entrenar desde cero es barato y rápido** (segundos).
- **Conclusión:** arquitectura **MLP entrenada desde cero**, con regularización (Dropout, BatchNorm, L2, EarlyStopping) para compensar
  el tamaño moderado del conjunto.

## 6. Diseño y entrenamiento del modelo
| Componente | Elección | Justificación |
|---|---|---|
| Capas ocultas | 128 → 64 → 32 neuronas | Embudo: de combinar muchas features a representaciones compactas |
| Activación | ReLU | Evita el desvanecimiento del gradiente y es eficiente |
| Salida | Dense(3) + **Softmax** | Distribución de probabilidad sobre 3 clases mutuamente excluyentes |
| Pérdida | `sparse_categorical_crossentropy` | Etiquetas enteras (0,1,2), sin necesidad de one-hot en `y` |
| Optimizador | Adam, lr = 1e-3 | Adaptativo, robusto con poco ajuste |
| Regularización | BatchNorm + Dropout (0.4/0.3/0.2) + L2(1e-4) | Controlan el sobreajuste |
| Callbacks | `MejorModelo` (restaura los mejores pesos) + ReduceLROnPlateau | Entrena las 200 épocas completas y se queda con el modelo de menor `val_loss`; baja el lr cuando se estanca |


In [ ]:
def build_model(n_features, n_classes):
    reg = regularizers.l2(1e-4)
    model = keras.Sequential([
        layers.Input(shape=(n_features,)),
        # Bloque 1
        layers.Dense(128, kernel_regularizer=reg), layers.BatchNormalization(),
        layers.Activation("relu"), layers.Dropout(0.4),
        # Bloque 2
        layers.Dense(64, kernel_regularizer=reg), layers.BatchNormalization(),
        layers.Activation("relu"), layers.Dropout(0.3),
        # Bloque 3
        layers.Dense(32, kernel_regularizer=reg), layers.BatchNormalization(),
        layers.Activation("relu"), layers.Dropout(0.2),
        # Salida: probabilidades por clase
        layers.Dense(n_classes, activation="softmax"),
    ], name="mlp_desercion")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

model = build_model(Xtr.shape[1], len(le.classes_))
model.summary()

In [ ]:
EPOCHS = 200   # se entrenan TODAS las épocas (no hay parada anticipada)

class MejorModelo(keras.callbacks.Callback):
    """
    - Calcula en CADA época métricas extra sobre validación (F1 macro y Recall de Dropout).
    - Guarda en memoria los pesos de la época con menor val_loss y los RESTAURA al terminar,
      sin detener el entrenamiento: corren las 200 épocas, pero el modelo final es el mejor.
    """
    def __init__(self, X_val, y_val, i_drop, verbose=True):
        super().__init__()
        self.Xv, self.yv, self.i_drop, self.verbose = X_val, y_val, i_drop, verbose
        self.extra = {"val_f1_macro": [], "val_recall_dropout": []}
        self.best, self.best_epoch, self.best_w = np.inf, -1, None

    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.Xv, verbose=0).argmax(1)
        self.extra["val_f1_macro"].append(f1_score(self.yv, p, average="macro"))
        self.extra["val_recall_dropout"].append(
            recall_score(self.yv, p, labels=[self.i_drop], average=None, zero_division=0)[0])
        if logs["val_loss"] < self.best:
            self.best, self.best_epoch, self.best_w = logs["val_loss"], epoch, self.model.get_weights()

    def on_train_end(self, logs=None):
        self.model.set_weights(self.best_w)
        if self.verbose:
            print(f"\n>> Pesos restaurados de la época {self.best_epoch + 1} (menor val_loss = {self.best:.4f})")

i_drop = list(le.classes_).index("Dropout")
mejor = MejorModelo(Xva, y_val, i_drop)
callbacks = [
    mejor,
    # Baja el learning rate cuando val_loss se estanca: así las métricas siguen cambiando época tras época
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=8, min_lr=1e-6, verbose=1),
]

history = model.fit(
    Xtr, y_train,
    validation_data=(Xva, y_val),
    epochs=EPOCHS, batch_size=32,
    class_weight=class_weight,        # corrige el desbalance
    callbacks=callbacks, verbose=2,
)
print("Épocas ejecutadas:", len(history.history["loss"]), "de", EPOCHS)

In [ ]:
# Curvas de aprendizaje: Train vs Validation (todas las épocas) + métricas extra de validación
h = history.history
ep = np.arange(1, len(h["loss"]) + 1)
be = mejor.best_epoch + 1
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].plot(ep, h["loss"], label="Train"); axes[0].plot(ep, h["val_loss"], label="Validation")
axes[0].set_title("Pérdida (loss)")
axes[1].plot(ep, h["accuracy"], label="Train"); axes[1].plot(ep, h["val_accuracy"], label="Validation")
axes[1].set_title("Exactitud (accuracy)")
axes[2].plot(ep, mejor.extra["val_f1_macro"], label="F1 macro (val)")
axes[2].plot(ep, mejor.extra["val_recall_dropout"], label="Recall Dropout (val)")
axes[2].set_title("Métricas de validación por época")
for a in axes:
    a.axvline(be, color="gray", ls="--", label=f"Mejor época ({be})"); a.set_xlabel("Época"); a.legend()
plt.tight_layout(); plt.savefig("curvas_entrenamiento.png", dpi=150); plt.show()

# Tabla de métricas por época (se muestran las primeras 5, la mejor y las últimas 5)
tab = pd.DataFrame({"epoca": ep, "loss": h["loss"], "acc": h["accuracy"], "val_loss": h["val_loss"],
                    "val_acc": h["val_accuracy"], "val_f1_macro": mejor.extra["val_f1_macro"],
                    "val_recall_dropout": mejor.extra["val_recall_dropout"], "lr": h["learning_rate"]})
display(pd.concat([tab.head(5), tab[tab.epoca == be], tab.tail(5)]).drop_duplicates().round(4))
print("Épocas en las que val_loss cambió respecto a la anterior:", int((tab.val_loss.diff().abs() > 0).sum()), "de", len(tab) - 1)

**Cómo interpretarlas:** se entrenan **las 200 épocas completas**; cada época actualiza Train/Validation y las métricas extra
(F1 macro y Recall de Dropout en validación). Se espera que *Train* siga mejorando mientras *Validation* toca su mínimo en las
primeras épocas y luego se estanca o empeora: eso es **sobreajuste**. Para que no afecte al modelo final, el callback `MejorModelo`
guarda los pesos de la época con menor `val_loss` (línea punteada) y los **restaura al terminar**; el modelo evaluado en test es ese.
Los saltos suaves de `loss` coinciden con `ReduceLROnPlateau` reduciendo el learning rate.

**Lectura para el informe:** se observa sobreajuste moderado (Train por encima de Validation). Dropout, BatchNorm y L2 lo frenan, y la
restauración de los mejores pesos garantiza que el modelo final no sufre las épocas posteriores. Con ~664 muestras por
conjunto la estimación tiene incertidumbre de varios puntos porcentuales; por eso se añade validación cruzada en la sección 7.

## 7. Evaluación sobre el conjunto de prueba (datos nunca vistos)

In [ ]:
probs_test = model.predict(Xte, verbose=0)
y_pred = probs_test.argmax(axis=1)
nombres = list(le.classes_)

print("Accuracy test:", round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred, target_names=nombres, digits=3))

for avg in ["macro", "weighted"]:
    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, average=avg)
    print(f"{avg:8s} -> Precision={p:.3f}  Recall={r:.3f}  F1={f:.3f}")

# Matriz de confusión (conteos y normalizada por fila = recall)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=nombres).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Matriz de confusión (conteos)")
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, normalize="true"), display_labels=nombres).plot(
    ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f")
axes[1].set_title("Normalizada por clase real (recall)")
plt.tight_layout(); plt.savefig("matriz_confusion.png", dpi=150); plt.show()

In [ ]:
# --- Enfoque en la clase Dropout (one-vs-rest) ---
i = nombres.index("Dropout")
tp = cm[i, i]; fn = cm[i, :].sum() - tp; fp = cm[:, i].sum() - tp
print(f"Dropout -> VP={tp}  FN={fn}  FP={fp}")
print(f"Recall    (de los que desertan, cuántos detectamos): {tp / (tp + fn):.3f}")
print(f"Precision (de los que alertamos, cuántos desertan de verdad): {tp / (tp + fp):.3f}")

### 7.1 Ajuste del umbral de decisión para Dropout (priorizar Recall)
Como el FN es el error más costoso, en lugar de afirmarlo sin evidencia lo **medimos**: la regla es «alertar Dropout si P(Dropout) ≥ t».
El umbral `t` se elige **solo con el conjunto de validación** (el menor sacrificio de Precision con Recall ≥ 0.85; si ninguno lo
logra, el que maximiza F2) y **después se reporta una única vez en test**, para no filtrar información del test.

In [ ]:
i_drop = list(le.classes_).index("Dropout")

def pred_thr(p, t):
    """Alerta Dropout si P(Dropout) >= t; si no, la mejor de las otras clases."""
    otras = p.copy(); otras[:, i_drop] = -1
    return np.where(p[:, i_drop] >= t, i_drop, otras.argmax(1))

def metr(y, yp):
    return (recall_score(y, yp, labels=[i_drop], average=None)[0],
            precision_score(y, yp, labels=[i_drop], average=None)[0],
            f1_score(y, yp, labels=[i_drop], average=None)[0],
            accuracy_score(y, yp))

pv = model.predict(Xva, verbose=0)
filas = [(round(t, 2), *metr(y_val, pred_thr(pv, t))) for t in np.arange(0.20, 0.61, 0.05)]
tabla = pd.DataFrame(filas, columns=["umbral", "recall_dropout", "precision_dropout", "f1_dropout", "accuracy"])
display(tabla.round(3))

# Selección en VALIDACIÓN
cand = tabla[tabla.recall_dropout >= 0.85]
if len(cand):
    THR = float(cand.sort_values("precision_dropout", ascending=False).iloc[0]["umbral"])
else:
    f2 = 5 * tabla.precision_dropout * tabla.recall_dropout / (4 * tabla.precision_dropout + tabla.recall_dropout)
    THR = float(tabla.loc[f2.idxmax(), "umbral"])
print(f"\nUmbral elegido (validación): t = {THR}")

# Reporte ÚNICO en TEST: argmax vs umbral
y_pred_thr = pred_thr(probs_test, THR)
for nombre_, yp in [("argmax (sin umbral)", y_pred), (f"umbral t={THR}", y_pred_thr)]:
    r, p_, f_, a = metr(y_test, yp)
    print(f"{nombre_:22s} -> Dropout: recall={r:.3f} precision={p_:.3f} F1={f_:.3f} | accuracy global={a:.3f}")
print()
print(classification_report(y_test, y_pred_thr, target_names=nombres, digits=3))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_thr), display_labels=nombres).plot(cmap="Blues", colorbar=False)
plt.title(f"Matriz de confusión en test con umbral Dropout = {THR}")
plt.tight_layout(); plt.savefig("matriz_confusion_umbral.png", dpi=150); plt.show()

**Cómo leerlo:** al bajar el umbral sube el Recall de Dropout (menos FN) a costa de Precision (más FP) y de algo de accuracy global.
Ese intercambio es justamente la decisión de negocio: aceptamos más falsas alarmas (baratas) para dejar escapar menos desertores (caros).
Los números exactos son los que imprime la celda anterior; cítalos en el informe.

### 7.2 Modelos de referencia (baselines) y validación cruzada
Un MLP solo se justifica si se compara con alternativas simples. Entrenamos Regresión Logística y Random Forest sobre los mismos
datos preprocesados y, además, estimamos la variabilidad con **validación cruzada estratificada de 5 pliegues** (el preprocesador se
reajusta dentro de cada pliegue, sin fuga de datos).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

# --- Baselines en el mismo split ---
res = [("MLP (Keras)", accuracy_score(y_test, y_pred), f1_score(y_test, y_pred, average="macro"),
        recall_score(y_test, y_pred, labels=[i_drop], average=None)[0])]
for n, c in [("Regresión logística", LogisticRegression(max_iter=2000, class_weight="balanced")),
             ("Random Forest", RandomForestClassifier(300, class_weight="balanced", random_state=SEED, n_jobs=-1))]:
    c.fit(Xtr, y_train); p = c.predict(Xte)
    res.append((n, accuracy_score(y_test, p), f1_score(y_test, p, average="macro"),
                recall_score(y_test, p, labels=[i_drop], average=None)[0]))
display(pd.DataFrame(res, columns=["modelo", "accuracy_test", "F1_macro_test", "recall_Dropout_test"]).round(3))

# --- Validación cruzada 5-fold (sobre train+val, el test queda intacto) ---
X_cv = pd.concat([X_train, X_val]); y_cv = np.concatenate([y_train, y_val])
EPOCHS_CV = 100   # menos épocas que el entrenamiento principal para que la CV sea rápida (misma lógica: mejores pesos)
accs, f1s, recs = [], [], []
for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X_cv, y_cv)):
    pre_k = build_preprocessor()
    Xa = pre_k.fit_transform(X_cv.iloc[a]).astype("float32"); Xb = pre_k.transform(X_cv.iloc[b]).astype("float32")
    keras.backend.clear_session(); tf.random.set_seed(SEED + k)
    wk = compute_class_weight("balanced", classes=np.unique(y_cv[a]), y=y_cv[a])
    m = build_model(Xa.shape[1], len(le.classes_))
    Xa2, Xv2, ya2, yv2 = train_test_split(Xa, y_cv[a], test_size=0.15, stratify=y_cv[a], random_state=SEED)
    cb = MejorModelo(Xv2, yv2, i_drop, verbose=False)
    m.fit(Xa2, ya2, validation_data=(Xv2, yv2), epochs=EPOCHS_CV, batch_size=32,
          class_weight={int(c): float(w) for c, w in enumerate(wk)}, verbose=0, callbacks=[cb])
    pk = m.predict(Xb, verbose=0).argmax(1)
    accs.append(accuracy_score(y_cv[b], pk)); f1s.append(f1_score(y_cv[b], pk, average="macro"))
    recs.append(recall_score(y_cv[b], pk, labels=[i_drop], average=None)[0])
print(f"CV 5-fold MLP -> accuracy {np.mean(accs):.3f} ± {np.std(accs):.3f} | F1 macro {np.mean(f1s):.3f} ± {np.std(f1s):.3f} "
      f"| recall Dropout {np.mean(recs):.3f} ± {np.std(recs):.3f}")

**Interpretación honesta:** en datos tabulares pequeños es habitual que la regresión logística o el Random Forest rindan **parecido** al
MLP. Si la tabla anterior lo muestra, se reporta así: el MLP cumple lo que pide el enunciado (red neuronal con regularización, Softmax,
pérdida y optimizador justificados) y su rendimiento es competitivo, pero no se afirma que sea superior sin evidencia. La desviación
estándar de la validación cruzada indica cuánta incertidumbre tiene la cifra de accuracy de un único split (~664 casos de test).
La clase *Enrolled* tiene el F1 más bajo (≈ 0.57) porque es un estado **intermedio** entre abandonar y graduarse, con perfiles que se
solapan con las otras dos clases.

### Análisis de negocio: ¿qué error es más costoso?
**Falso Negativo (FN)** = predecir "continúa/se gradúa" cuando en realidad **desertará**. **Falso Positivo (FP)** = predecir "desertará"
cuando en realidad **continuará**.

- **El FN es el más costoso.** Es un estudiante en riesgo **que el sistema no detecta**: no recibe tutoría, apoyo económico ni
  acompañamiento psicosocial y abandona. La institución pierde la matrícula futura (ingresos), el esfuerzo invertido en
  captación y acompañamiento, empeora sus indicadores de retención/acreditación y, sobre todo, el estudiante pierde una oportunidad
  educativa. Además el daño es **irreversible**: la alerta llega tarde o nunca.
- **El FP es un error barato y recuperable.** Un estudiante estable recibe una intervención preventiva que quizá no
  necesitaba (una llamada, una tutoría). El costo es de recursos de bienestar, limitados pero acotados, y puede incluso ser beneficioso.
- **Implicación:** para Dropout se debe **priorizar el Recall** (minimizar FN), aceptando algo menos de Precision. Por eso en 7.1
  se baja el umbral de decisión (alertar si P(Dropout) ≥ t, con `t` elegido en validación) y se vigila el **F1 de Dropout** como
  equilibrio; la alternativa sería subir el peso de esa clase en la pérdida. La clase *Enrolled* es la más difícil de separar porque
  es un estado intermedio entre las otras dos.


## 8. Exportación del modelo y de los artefactos de preprocesamiento

In [ ]:
os.makedirs("artifacts", exist_ok=True)
model.save("artifacts/model.keras")
joblib.dump(pre, "artifacts/preprocessor.joblib")       # imputadores + OneHotEncoder + Winsorizer + StandardScaler
joblib.dump(le, "artifacts/label_encoder.joblib")       # mapeo 0/1/2 <-> Dropout/Enrolled/Graduate

# Metadatos para construir el formulario de la app (valores por defecto, rangos, opciones)
meta = {"classes": list(map(str, le.classes_)), "defaults": {}, "ranges": {}, "options": {},
        "dropout_threshold": THR,                      # umbral elegido en validación (sección 7.1)
        "versions": {"tensorflow": tf.__version__, "scikit-learn": sklearn.__version__}}
for c in CATEGORICAL + BINARY:
    meta["defaults"][c] = float(X_train[c].mode().iloc[0])
for c in CONTINUOUS:
    med = float(X_train[c].median())
    meta["defaults"][c] = med
    meta["ranges"][c] = {"min": float(X_train[c].min()), "max": float(X_train[c].max()), "median": med}
for c in CATEGORICAL:
    meta["options"][c] = sorted(float(v) for v in X_train[c].dropna().unique())
with open("artifacts/metadata.json", "w") as f:
    json.dump(meta, f)

# Verificación: recargar y comprobar que las predicciones coinciden
m2, pre2 = keras.models.load_model("artifacts/model.keras"), joblib.load("artifacts/preprocessor.joblib")
check = m2.predict(pre2.transform(X_test.iloc[:5]).astype("float32"), verbose=0)
print("¿Predicciones idénticas tras recargar?", np.allclose(check, probs_test[:5], atol=1e-5))
print(os.listdir("artifacts"))

In [ ]:
# Pruebas de cordura (3 perfiles) con el mismo flujo que usa la app: valores por defecto + perfil
def predecir(perfil):
    fila = dict(meta["defaults"]); fila.update(perfil)
    p = m2.predict(pre2.transform(to_numeric_frame(pd.DataFrame([fila]))).astype("float32"), verbose=0)[0]
    return {c: round(float(v), 3) for c, v in zip(le.classes_, p)}

alto = {"Curricular units 1st sem (approved)": 6, "Curricular units 1st sem (grade)": 15,
        "Curricular units 2nd sem (approved)": 6, "Curricular units 2nd sem (grade)": 15,
        "Debtor": 0, "Tuition fees up to date": 1, "Scholarship holder": 1}
riesgo = {"Curricular units 1st sem (approved)": 0, "Curricular units 1st sem (grade)": 0,
          "Curricular units 2nd sem (approved)": 0, "Curricular units 2nd sem (grade)": 0,
          "Debtor": 1, "Tuition fees up to date": 0, "Scholarship holder": 0}
print("Notas altas, al día, becario :", predecir(alto))
print("Sin aprobadas, deudor, mora  :", predecir(riesgo))
print("Perfil típico (defaults)     :", predecir({}))

## 9. Aplicación web con Streamlit (`app.py`)
La app carga los artefactos, muestra un formulario con las variables clave, completa el resto con valores típicos del
entrenamiento, aplica **el mismo** `preprocessor.joblib` y muestra la clase predicha con la confianza (Softmax) en barras.

In [ ]:
%%writefile app.py
"""
app.py - Aplicación Streamlit para predecir deserción estudiantil.
Requiere la carpeta ./artifacts (model.keras, preprocessor.joblib, label_encoder.joblib, metadata.json)
y el módulo preprocessing.py en la misma carpeta.
"""
import json
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st
from tensorflow import keras

from preprocessing import to_numeric_frame  # importar también registra la clase Winsorizer para joblib

BASE = Path(__file__).parent / "artifacts"

# En este CSV de Kaggle los programas vienen recodificados como 1..17 (orden de los códigos originales del UCI).
# Verifica 2-3 casos con tu criterio/fuente si vas a citar nombres de programa en el informe.
COURSES = {
    1: "Biofuel Production Technologies", 2: "Animation and Multimedia Design",
    3: "Social Service (evening)", 4: "Agronomy", 5: "Communication Design",
    6: "Veterinary Nursing", 7: "Informatics Engineering", 8: "Equinculture",
    9: "Management", 10: "Social Service", 11: "Tourism", 12: "Nursing",
    13: "Oral Hygiene", 14: "Advertising and Marketing Management",
    15: "Journalism and Communication", 16: "Basic Education",
    17: "Management (evening)",
}
LABELS_ES = {"Dropout": "Deserción (Dropout)", "Enrolled": "Sigue matriculado (Enrolled)",
             "Graduate": "Graduado (Graduate)"}

st.set_page_config(page_title="Riesgo de deserción", page_icon="🎓", layout="wide")


@st.cache_resource
def load_artifacts():
    model = keras.models.load_model(BASE / "model.keras")
    pre = joblib.load(BASE / "preprocessor.joblib")
    le = joblib.load(BASE / "label_encoder.joblib")
    meta = json.loads((BASE / "metadata.json").read_text())
    return model, pre, le, meta


model, pre, le, meta = load_artifacts()


# ---------- helpers de widgets ----------
def num(label, col, step=1.0):
    r = meta["ranges"][col]
    return st.number_input(label, min_value=r["min"], max_value=r["max"],
                           value=r["median"], step=step, key=col)


def yesno(label, col):
    return st.radio(label, [0, 1], index=int(meta["defaults"][col]), horizontal=True,
                    format_func=lambda v: "Sí" if v == 1 else "No", key=col)


def cat(label, col, names=None):
    opts = meta["options"][col]
    fmt = (lambda v: f"{int(v)} - {names.get(int(v), '')}") if names else (lambda v: str(int(v)))
    return st.selectbox(label, opts, index=opts.index(meta["defaults"][col]),
                        format_func=fmt, key=col)


# ---------- interfaz ----------
st.title("🎓 Predicción de deserción estudiantil")
st.caption("Red neuronal (MLP) entrenada con 'Predict Students' Dropout and Academic Success'. "
           "Las variables que no aparecen en el formulario se completan con la mediana/moda del conjunto de entrenamiento.")

with st.form("form_estudiante"):
    st.subheader("1. Datos personales y socioeconómicos")
    c1, c2, c3, c4 = st.columns(4)
    user = {}
    with c1:
        user["Age at enrollment"] = num("Edad al matricularse", "Age at enrollment")
        user["Gender"] = st.radio("Género", [0, 1], index=int(meta["defaults"]["Gender"]), horizontal=True,
                                  format_func=lambda v: "Masculino" if v == 1 else "Femenino", key="Gender")
    with c2:
        user["Marital status"] = cat("Estado civil (código)", "Marital status")
        user["Displaced"] = yesno("¿Desplazado?", "Displaced")
    with c3:
        user["Scholarship holder"] = yesno("¿Tiene beca?", "Scholarship holder")
        user["Debtor"] = yesno("¿Es deudor?", "Debtor")
    with c4:
        user["Tuition fees up to date"] = yesno("¿Matrícula al día?", "Tuition fees up to date")
        user["Daytime/evening attendance"] = st.radio(
            "Jornada", [1, 0], index=0 if meta["defaults"]["Daytime/evening attendance"] == 1 else 1,
            horizontal=True, format_func=lambda v: "Diurna" if v == 1 else "Nocturna", key="Daytime/evening attendance")

    st.subheader("2. Admisión y programa")
    c1, c2 = st.columns(2)
    with c1:
        user["Course"] = cat("Programa", "Course", COURSES)
    with c2:
        user["Application mode"] = cat("Modo de admisión (código)", "Application mode")

    st.subheader("3. Rendimiento académico")
    for sem, tag in (("1st", "Primer semestre"), ("2nd", "Segundo semestre")):
        st.markdown(f"**{tag}**")
        cols = st.columns(5)
        items = [("enrolled", "Materias inscritas"), ("evaluations", "Evaluaciones"),
                 ("approved", "Materias aprobadas"), ("grade", "Promedio"),
                 ("without evaluations", "Sin evaluaciones")]
        for col_ui, (key, label) in zip(cols, items):
            name = f"Curricular units {sem} sem ({key})"
            with col_ui:
                user[name] = num(label, name, 0.5 if key == "grade" else 1.0)

    enviado = st.form_submit_button("Predecir", type="primary")

if enviado:
    # 1) completar con valores por defecto, 2) mismo preprocesamiento que en entrenamiento
    row = dict(meta["defaults"])
    row.update(user)
    X = pre.transform(to_numeric_frame(pd.DataFrame([row]))).astype("float32")
    probs = model.predict(X, verbose=0)[0]
    classes = list(le.classes_)
    thr = float(meta.get("dropout_threshold", 0.5))
    i_drop = classes.index("Dropout")
    # Regla de negocio: alerta de Dropout si P(Dropout) >= umbral (elegido en validación); si no, la mejor de las otras clases
    if probs[i_drop] >= thr:
        best = i_drop
    else:
        otras = probs.copy(); otras[i_drop] = -1
        best = int(otras.argmax())

    st.divider()
    c1, c2 = st.columns([1, 2])
    with c1:
        st.metric("Clase predicha", LABELS_ES.get(classes[best], classes[best]))
        st.metric("Nivel de confianza (probabilidad de la clase)", f"{probs[best] * 100:.1f}%")
        st.caption(f"Umbral de alerta Dropout: P ≥ {thr:.2f}")
        if classes[best] == "Dropout":
            st.error("⚠️ Estudiante en riesgo de deserción: se recomienda intervención temprana.")
        elif classes[best] == "Enrolled":
            st.info("Estudiante probablemente continuará matriculado.")
        else:
            st.success("Estudiante con alta probabilidad de graduarse.")
    with c2:
        st.subheader("Probabilidades (Softmax)")
        chart = pd.DataFrame({"Probabilidad (%)": probs * 100}, index=[LABELS_ES.get(c, c) for c in classes])
        st.bar_chart(chart)
        for c, p in zip(classes, probs):
            st.progress(float(p), text=f"{c}: {p * 100:.1f}%")

### Ejecutar la app dentro de Colab
Colab no expone puertos directamente, así que se usa un **túnel gratuito de Cloudflare** (no requiere cuenta ni token).
Al ejecutar la celda aparecerá un enlace `https://….trycloudflare.com`; ábrelo en otra pestaña. Es temporal: dura mientras la celda siga activa.

In [ ]:
import subprocess, time, re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!pkill -f "[s]treamlit run" ; pkill -f "[c]loudflared tunnel" ; true

# 1) Levantar Streamlit en segundo plano
st_proc = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                           stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)

# 2) Abrir el túnel y capturar la URL pública
cf = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                      stderr=subprocess.PIPE, text=True)
url = None
for linea in cf.stderr:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", linea)
    if m:
        url = m.group(0); break
print("🌐 Abre la app aquí:", url)

### Descargar el proyecto (para subir a GitHub)

> **Para la sustentación:** lo más seguro es correr la app **localmente** (`streamlit run app.py`, instrucciones en el README del zip).
> El túnel de Cloudflare de Colab es temporal y puede fallar; úsalo solo como respaldo. Descarga también este cuaderno
> (*Archivo → Descargar .ipynb*) y colócalo en el repositorio.

In [ ]:
import shutil
from google.colab import files

os.makedirs("proyecto_desercion", exist_ok=True)
shutil.copytree("artifacts", "proyecto_desercion/artifacts", dirs_exist_ok=True)
for f in ["app.py", "preprocessing.py", "curvas_entrenamiento.png", "matriz_confusion.png", "matriz_confusion_umbral.png"]:
    shutil.copy(f, "proyecto_desercion/")

# Versiones fijadas: los .keras / .joblib son sensibles a la versión de TensorFlow / scikit-learn
open("proyecto_desercion/requirements.txt", "w").write(
    f"streamlit\ntensorflow=={tf.__version__}\nscikit-learn=={sklearn.__version__}\n"
    f"pandas=={pd.__version__}\nnumpy=={np.__version__}\njoblib=={joblib.__version__}\n")

open("proyecto_desercion/.gitignore", "w").write("__pycache__/\n*.pyc\n.ipynb_checkpoints/\nstreamlit.log\ncloudflared\n.venv/\n")

open("proyecto_desercion/README.md", "w").write(f"""# Predicción de deserción estudiantil (MLP + Streamlit)

Dataset: *Predict Students' Dropout and Academic Success* (Kaggle). Modelo: MLP (Keras) entrenado desde cero.
Umbral de alerta Dropout (elegido en validación): **{THR}**.

## Contenido
- `Parcial_Desercion_Estudiantil.ipynb`: preparación de datos, EDA, entrenamiento, evaluación, exportación.
- `preprocessing.py`: preprocesamiento único (entrenamiento + app).
- `app.py`: app Streamlit (formulario -> clase predicha + confianza).
- `artifacts/`: `model.keras`, `preprocessor.joblib`, `label_encoder.joblib`, `metadata.json`.

## Ejecutar localmente (recomendado para la sustentación)
```bash
python -m venv .venv && source .venv/bin/activate   # Windows: .venv\\Scripts\\activate
pip install -r requirements.txt
streamlit run app.py
```
Las versiones de `requirements.txt` son las usadas al entrenar (los archivos `.keras` y `.joblib` dependen de ellas).
""")
# Incluir el cuaderno si se subió/guardó en el directorio de trabajo
for nbf in glob.glob("*.ipynb"):
    shutil.copy(nbf, "proyecto_desercion/")
shutil.make_archive("proyecto_desercion", "zip", "proyecto_desercion")
files.download("proyecto_desercion.zip")
